# 06 — Loss, Chain Rule, Backpropagation, and Gradient Checking (TensorFlow / Keras)


> **Learning contract.** Every code cell is preceded by an explanation of what the code does, why the operation exists mathematically, what tensor/array shapes are expected, and what production or business failure it prevents. Run the notebooks in numerical order in a fresh Conda environment.


Loss converts prediction quality into an optimization objective. For one correct class with predicted probability $p_y$, cross-entropy is $L=-\log p_y$. Backpropagation does **not** "send the error backward" as a number; it applies the chain rule to compute each parameter's sensitivity $\partial L/\partial \theta$.

For a sigmoid scalar example $z=wx+b$, $a=\sigma(z)$ and BCE with target 1, $\partial L/\partial z=a-1$, so $\partial L/\partial w=(a-1)x$.


## Code walkthrough — manual derivative and finite-difference verification
The analytical derivative is compared to the central-difference approximation. Gradient checking is slow and not used for training, but it is a powerful debugging technique for custom layers and manual backprop implementations.


In [1]:
import numpy as np

x = 0.8
w = 0.5
b = -0.15
y = 1.0


def scalar_loss(w_value):
    z = w_value * x + b
    a = 1 / (1 + np.exp(-z))
    return -(y * np.log(a) + (1 - y) * np.log(1 - a))


z = w * x + b
a = 1 / (1 + np.exp(-z))
analytic = (a - y) * x
eps = 1e-05
numeric = (scalar_loss(w + eps) - scalar_loss(w - eps)) / (2 * eps)
print("z", z, "a", a, "loss", scalar_loss(w))
print(
    "analytic dL/dw",
    analytic,
    "finite-difference",
    numeric,
    "absolute error",
    abs(analytic - numeric),
)

z 0.25 a 0.5621765008857981 loss 0.5759394198788437
analytic dL/dw -0.35025879929136156 finite-difference -0.350258799286518 absolute error 4.8435699895321704e-12


## Code walkthrough — map the same derivative into TensorFlow / Keras
This cell shows what the framework automates. The local mathematics is unchanged; only gradient bookkeeping changes. Inspect the resulting gradient values instead of treating `backward()` or `GradientTape` as magic.


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

tf.random.set_seed(42)
x = tf.constant([[0.8, 0.1, 0.6, 0.0]], dtype=tf.float32)
w = tf.Variable([[0.5], [-0.3], [0.2], [0.7]], dtype=tf.float32)
b = tf.Variable([-0.15], dtype=tf.float32)
with tf.GradientTape() as tape:
    z = x @ w + b
    a = tf.sigmoid(z)
    loss = -tf.math.log(a)
dw, db = tape.gradient(loss, [w, b])
print("GradientTape dw=", dw.numpy().ravel(), "db=", db.numpy(), "loss=", float(loss))

2026-09-07 18:54:51.106176: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


GradientTape dw= [-0.33264762 -0.04158095 -0.24948572 -0.        ] db= [-0.4158095] loss= 0.5375281572341919


## Chain-rule map for the two-layer MLP
`loss → dLogits → dW2/db2 → dA1 → dZ1 through activation derivative → dW1/db1`. Every arrow multiplies or contracts a local derivative with an upstream gradient. Shape checking is one of the best ways to debug backprop.


## Business implication
A model can produce plausible predictions while gradients are wrong, especially in custom objectives or layers. Gradient verification and loss sanity checks reduce the risk of silently training the wrong objective.
